# Project 81 — Local OCR + RAG Assistant
## OCR Extraction → Chunking → Vector Search → Q&A

**Stack:** LangChain · Ollama · ChromaDB · Pillow · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb Pillow

## Step 1 — Create Sample Document Image (Simulated OCR)

In [ ]:
from pathlib import Path

Path("sample_data").mkdir(exist_ok=True)

# Simulate OCR output (in production, use pytesseract or PaddleOCR)
ocr_texts = {
    "page_1.txt": """INVOICE
Invoice Number: INV-2024-0042
Date: January 15, 2024
Bill To: Acme Corporation, 123 Business Ave, Suite 400

Item                    Qty    Price     Total
Cloud Hosting (Annual)   1    $12,000   $12,000
Premium Support          1    $3,000    $3,000
Data Migration Service   1    $5,000    $5,000

Subtotal: $20,000
Tax (8%): $1,600
Total Due: $21,600
Payment Terms: Net 30""",

    "page_2.txt": """SERVICE AGREEMENT
This Service Level Agreement (SLA) between Acme Corporation
and CloudTech Solutions establishes the terms of service.

Uptime Guarantee: 99.9% monthly uptime
Response Time: Critical issues within 1 hour
Support Hours: 24/7 for critical, business hours for standard
Data Retention: 7 years minimum
Backup Frequency: Daily incremental, weekly full
Disaster Recovery: RTO 4 hours, RPO 1 hour""",

    "page_3.txt": """QUARTERLY REPORT - Q4 2024
Revenue: $2.3M (up 15% QoQ)
Active Customers: 450 (up 12%)
Churn Rate: 2.1% (down from 3.4%)
NPS Score: 72 (up 8 points)
Key Wins: Enterprise deal with GlobalCorp ($500K ARR)
Challenges: Increased competition in mid-market segment
Next Quarter Focus: Product-led growth initiative""",
}

for filename, content in ocr_texts.items():
    Path(f"sample_data/{filename}").write_text(content, encoding="utf-8")
    print(f"Created: {filename} ({len(content)} chars)")

## Step 2 — Build Vector Store from OCR Text

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import shutil

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Load OCR text as documents
docs = []
for filename, content in ocr_texts.items():
    docs.append(Document(
        page_content=content,
        metadata={"source": filename, "type": filename.split("_")[0]}
    ))

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

shutil.rmtree("chroma_ocr", ignore_errors=True)
store = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_ocr")
retriever = store.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks from {len(docs)} OCR pages")

## Step 3 — Q&A Over OCR Documents

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer based on the OCR-extracted document context. Cite the source page."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

questions = [
    "What is the total amount due on the invoice?",
    "What is the uptime guarantee in the SLA?",
    "What was the revenue in Q4 2024?",
    "What is the disaster recovery RTO?",
    "How many active customers are there?",
]

for q in questions:
    docs_found = retriever.invoke(q)
    context = "\n---\n".join([f"[{d.metadata['source']}] {d.page_content}" for d in docs_found])
    answer = qa_chain.invoke({"context": context, "question": q})
    print(f"Q: {q}")
    print(f"A: {answer[:200]}")
    print()

## What You Learned
- **OCR-to-RAG pipeline** for scanned documents
- **Document chunking** with source tracking
- **Cited Q&A** over extracted text